# Reproducing Perich, Gallego & Miller (2018) from dandiset 000688

**Paper.** Perich MG, Gallego JA, Miller LE (2018). A neural population mechanism for rapid
learning. *Neuron* 100, 964-976.
[doi:10.1016/j.neuron.2018.09.030](https://doi.org/10.1016/j.neuron.2018.09.030). The claims file
was written from the accepted author manuscript in PubMed Central (PMC6250582), with its eight
supplement files (Figures S1 to S7 and their legends); this attempt read the same text.

**Dandiset.** [DANDI:000688](https://dandiarchive.org/dandiset/000688/0.250122.1735), version
0.250122.1735. The paper cites no DANDI version (it predates the dandiset). This is the only
published version; the draft has the same 111 files and was modified two days later, so the
published version is pinned.

**What the dandiset holds.** 111 sessions of four monkeys (C, J, M, T) in center-out (CO) and
random-target (RT) reaching, with sorted units, cursor kinematics and a trial table. The trial
tables have no column for the epoch (Baseline, Force or Rotation, Washout) or the perturbation,
and none of the files is described as a curl field or visuomotor rotation session. The dual-array
(M1 and PMd) center-out sessions of Monkeys C (2016) and M (February and March 2014) have unit
counts that match the paper's ranges, except one Monkey C session with too few PMd units, but each holds about 200 reward trials with small, steady
reach errors: the size of one Baseline epoch, without the Force or Rotation and Washout epochs.
The data check below shows this. Only the claims that use Baseline trials alone can be tested.

| Claim | Tested here |
|---|---|
| methods-neuron-counts | tested |
| fig3b-pmd-higher-dimensional | tested (Baseline trials, as the method uses) |
| figs4d-dimensionality-robust-to-size | tested |
| figs4c-null-potent-index-centered | tested (subspaces from Baseline, as the method uses) |
| fig6c-null-leads-potent | tested for Baseline; the end-of-Force part cannot be |
| the 23 other claims (behavior, covariance across epochs, GLMs, Washout, Mahalanobis distance, VR) | blocked, data missing |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import itertools
import time
import warnings
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score

import perich

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
urls = perich.asset_urls()
with ThreadPoolExecutor(16) as ex:
    meta = dict(zip(urls, ex.map(perich.read_metadata, urls.values())))
survey = pd.DataFrame([{"session": k, "monkey": k[4], "task": k.split("_")[1].split("-")[1],
                        "date": k.split("-")[-1], **v} for k, v in meta.items()]).sort_values("session")
print(f"read the metadata of {len(survey)} files of dandiset {perich.DANDISET_ID} version "
      f"{perich.DANDISET_VERSION} in {time.time() - started:.0f} s")

SESSIONS = perich.CANDIDATE_SESSIONS
sessions = perich.load_sessions(SESSIONS, urls)
by_id = {s["sid"]: s for s in sessions}
print(f"loaded spikes, kinematics and trials of {len(sessions)} sessions in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print("Files per monkey and task:")
print(pd.crosstab(survey.monkey, survey.task).to_string())
print("\nTrial table columns (sets of columns and how many files have each):")
print(survey.trial_columns.apply(", ".join).value_counts().to_string())
print("\nExperiment descriptions:", survey.experiment.value_counts().to_dict())
print("\nSession descriptions (text, number of files):")
print(survey.session_description.value_counts().to_string())
warned = survey[survey.session_description.str.contains("Warning")]
print("Files whose session description holds a warning:", list(warned.session))
survey["arrays"] = np.where((survey.n_M1 > 0) & (survey.n_PMd > 0), "M1+PMd", np.where(survey.n_PMd > 0, "PMd", "M1"))
print("\nArrays per monkey (sessions):")
print(pd.crosstab(survey.monkey, survey.arrays).to_string())
survey["rewards"] = survey.results.apply(lambda r: r.get("R", 0))
fp = survey.groupby(["n_trials", "n_M1", "n_PMd", "n_spikes"]).session.apply(list)
print("\nFiles sharing trial count, unit counts per area and total spike count:", [v for v in fp if len(v) > 1] or "none")
print("\nAll files, with units per area and trial outcomes:")
print(survey[["session", "task", "arrays", "n_M1", "n_PMd", "n_trials", "rewards", "last_stop"]].to_string(index=False))

No trial table has a column for the epoch or the perturbation. The experiment descriptions say only
"center-out" or "random target", and the session descriptions only which monkey performed which
task, except in seven Monkey C sessions (2016-09-09 to 2016-09-23), where the text ends in
"Warning: This session with Monkey": the warning is cut off in the file and in the archive's
metadata, so what it warns of is unknown. These are the seven sessions whose cursor positions do
not match the stored targets (below). The paper's sessions are center-out sessions of Monkeys C and M
with arrays in both M1 and PMd (STAR Methods, Neural recordings). Monkey C has such sessions only
from 2016-09-09 to 2016-10-21 (14). Monkey M has 7 from 2014-02-03 to 2014-03-07 and 13 later ones
(2014-09-29, 2014-12-03, two in May 2015 and nine in June 2015), with 10 to 30 M1 and 22 to 124 PMd
units and 174 to 733 rewards. Of the later ones, only 2015-05-12 (30 M1, 60 PMd) is inside the
paper's Monkey M ranges widened by 10%; the others have too few M1 units (2014-09-29 has 21) or too
few PMd units. The 7 February and March sessions match the paper's count of Monkey M sessions and
its bounds (66 and 121 PMd, 26 M1), so they are taken as the paper's, and 2015-05-12, with 443
rewards, is left out. That choice rests on the fit of the whole block, not on a rule applied to
each session: C 2016-10-13 and 2016-10-21 are also outside the Baseline reward range and are kept
among the Monkey C candidates (the second session set leaves them out). The 21 sessions of the two
February to March 2014 and September to October 2016 blocks are the candidates analysed here. No two files share their trial count, unit counts and
spike count, so no recording is deposited twice.

In [ ]:
PAPER = {"C": {"PMd": (137, 256), "M1": (55, 93)}, "M": {"PMd": (66, 121), "M1": (26, 51)}}
BASELINE_REWARDS = {"CF": (170, 225), "VR": (154, 217)}   # STAR Methods, Behavioral task
cand = []
for s in sessions:
    tr = s["trials"]
    cand.append({"session": s["sid"], "monkey": s["monkey"], "n_M1": int((s["area"] == "M1").sum()),
                 "n_PMd": int((s["area"] == "PMd").sum()), "trials": len(tr),
                 "rewards": int((tr.result == "R").sum()),
                 "minutes": round((tr.stop.iloc[-1] - tr.start.iloc[0]) / 60, 1)})
cand = pd.DataFrame(cand)
lo = min(v[0] for v in BASELINE_REWARDS.values()); hi = max(v[1] for v in BASELINE_REWARDS.values())
cand["in_baseline_range"] = cand.rewards.between(lo, hi)
print(cand.to_string(index=False))
print(f"\nPaper: Baseline 170-225 rewards (CF) or 154-217 successful trials (VR); Force 201-337, Rotation 219-316; "
      f"Washout 153-404 (CF) or 162-348 (VR). A whole CF session therefore has at least {170+201+153} rewards.")
print("Reward trials per candidate session:", cand.rewards.min(), "to", cand.rewards.max(),
      f"; within the Baseline range {lo}-{hi}:", int(cand.in_baseline_range.sum()), "of", len(cand))
SETS = {"all 21": list(cand.session), "19 within Baseline reward range": list(cand.session[cand.in_baseline_range])}
for k, v in SETS.items():
    print(f"session set '{k}':", len(v), "sessions;", pd.Series([x[4] for x in v]).value_counts().to_dict())

Every candidate session has 154 to 286 reward trials, while a whole curl field session of the paper
has at least 524 (Baseline, Force and Washout together). 19 of the 21 fall inside the paper's
Baseline range (154 to 225); C 2016-10-13 (239) and C 2016-10-21 (286) do not. The files therefore
hold one epoch each, consistent with the Baseline epoch only; the kinematics below check that no
block of perturbed reaches is present. The paper has 16 sessions (Monkey C five CF and four VR,
Monkey M four CF and three VR). The 7 Monkey M sessions match its count; Monkey C has 14
candidates for 9, and nothing in the files says which are the paper's or which were CF or VR
sessions. Two session sets are used throughout: all 21 candidates, and the 19 within the
Baseline reward range.

In [ ]:
rows = []
for s in sessions:
    tr = s["trials"]; r = tr[tr.result == "R"]
    t, pos = s["pos_t"], s["pos"]
    end = pos[np.clip(np.searchsorted(t, r.stop.values), 0, len(t) - 1)]
    errs = perich.initial_errors(s)
    errs["dev"] = errs.error - errs.groupby("target_id").error.transform("mean")
    rho_s = stats.spearmanr(np.arange(len(errs)), errs.dev)
    # end of movement: last sample before the stop at which speed exceeds 20% of the trial's peak,
    # compared with the assumed acquisition time (stop - 0.5 s hold); finite speeds only
    with np.errstate(all="ignore"):
        speed = np.hypot(*np.gradient(pos, t, axis=0).T)
    move_end = []
    for go, stop, acq in zip(r.go_cue, r.stop, r.acquire):
        a, b = np.searchsorted(t, [go, stop])
        sp = speed[a:b]; ok = np.isfinite(sp)
        if ok.sum() < 5:
            continue
        fast = np.flatnonzero(ok & (sp > 0.2 * sp[ok].max()))
        move_end.append(t[a + fast[-1]] - acq)
    run = errs.dev.abs().rolling(8).mean()
    rho = stats.spearmanr(np.arange(len(errs)), errs.dev.abs())
    rows.append({"session": s["sid"], "pos_unit": s["pos_unit"], "pos_hz": round(1 / np.median(np.diff(t))),
                 "repeated_timestamps": int((np.diff(t) <= 0).sum()),
                 "target_radius_cm": round(np.nanmedian(r.target_radius), 2),
                 "cursor_radius_at_stop": round(np.median(np.hypot(*end.T)), 2),
                 "peak_speed": round(errs.peak_speed.median(), 1),
                 "cursor_enters_target": round(r.cursor_enters_target.notna().mean(), 2),
                 "stop_minus_go_s": round((r.stop - r.go_cue).median(), 2),
                 "delay_s": f"{(r.go_cue - r.target_on).min():.2f}-{(r.go_cue - r.target_on).max():.2f}",
                 "move_end_minus_acquire_s": round(np.median(move_end), 2),
                 "signed_dev_first8": round(errs.dev[:8].mean(), 1), "signed_dev_last8": round(errs.dev[-8:].mean(), 1),
                 "rho_trial_signed": round(rho_s.statistic, 2), "p_rho_signed": round(rho_s.pvalue, 3),
                 "abs_dev_first8": round(errs.dev.abs()[:8].mean(), 1),
                 "abs_dev_last8": round(errs.dev.abs()[-8:].mean(), 1),
                 "max_8trial_mean": round(run.max(), 1), "median_abs_dev": round(errs.dev.abs().median(), 1),
                 "rho_trial_absdev": round(rho.statistic, 2), "p_rho": round(rho.pvalue, 3)})
kin = pd.DataFrame(rows)
print("Kinematics of reward trials (positions in the file's unit; errors in degrees, 150 ms after movement onset at 20% of peak speed,")
print("minus the session's mean per target):")
print(kin.to_string(index=False))
print("\nEnd of movement minus assumed acquisition (s), median over sessions:", kin.move_end_minus_acquire_s.median(),
      "; range", kin.move_end_minus_acquire_s.min(), "to", kin.move_end_minus_acquire_s.max())

Cursor positions are stated in cm and sampled at 100 Hz. The instructed delay spans about 0.5 to
1.5 s, as the Methods say, with a few longer delays in C 2016-09-09, C 2016-10-11 and M 2014-03-06
(up to 15.7 s). The reach errors show no block of perturbed reaches of the size a curl field or
rotation produces. The mean absolute deviation of the first eight reward trials is 3.9 to 14.0
degrees, against 2.9 to 9.4 for the last eight, outside two sessions. It is higher at the start by 3
to 8 degrees in M 2014-03-03 (14.0 against 6.4, signed 10.2 against 3.3, with a significant trend
in the signed deviation, p = 0.019), M 2014-02-03 (9.1 against 5.6), M 2014-02-18 (8.8 against
4.8) and C 2016-09-19 (9.5 against 6.5), and large throughout in C 2016-09-23 (25.3 and 41.0) and C 2016-10-11 (23.5 and 15.1). The last two come with
repeated cursor timestamps, which make the speed, and so the movement onset, undefined on some
trials (the count is in the table). Their signed deviations are 12.5 and 3.4 degrees over the first
eight trials and -4.6 and 1.2 over the last eight, with no significant trend in the signed
deviation (Spearman rho 0.12, p = 0.081, and 0.02, p = 0.784); in C 2016-10-11 the absolute
deviation decreases significantly over the session (rho -0.21, p = 0.003). Four sessions have a
significant trend in the signed deviation (C 2016-09-09, C 2016-10-05, M 2014-02-17, M 2014-03-03,
rho -0.14 to 0.22), with first and last eight-trial means within 10.2 degrees; rho is undefined for
M 2014-03-06. None shows the large offset that decays over tens of trials that the first trials of a
perturbation epoch would.

In the seven Monkey C sessions with the cut-off warning (2016-09-09 to 2016-09-23) the cursor ends
rewarded reaches about 4.2 cm from the center with a median peak speed near 14 cm/s, while the
target corners place every target at 8 cm and the later sessions end at about 7.9 cm with peak
speeds near 25 cm/s. Either the positions of those sessions are scaled differently or the target
corners are wrong; the paper's statement that Monkey C's first three sessions used 6 cm targets
does not fit either 4.2 or 8 cm. In the seven early Monkey C sessions the cursor never enters the square that
`target_corners` gives on a rewarded trial, and in M 2014-02-03 and M 2014-02-18 it enters it on 1%
of them (0.01 in the table). The analyses here use no positions; target acquisition is taken as the
trial's stop time minus the 0.5 s target hold (helper docstring). As a check of that choice, the
hand's speed last exceeds 20% of its peak a median of 0.09 to 0.15 s after this acquisition time
(per session), so the reach ends just after it, as when the cursor enters the target while still
slowing. This is consistent with a stop time at the end of the hold, but does not prove it.

In [ ]:
def decode(sess, align, t0, t1, rng):
    tr = sess["trials"]; r = tr[(tr.result == "R") & tr[align].notna()]
    X = np.stack([np.searchsorted(sp, r[align].values + t1) - np.searchsorted(sp, r[align].values + t0)
                  for sp in sess["spikes"]], axis=1)
    y = r.target_id.values
    cv = StratifiedKFold(5, shuffle=True, random_state=0)
    lda = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
    acc = cross_val_score(lda, X, y, cv=cv).mean()
    acc_shuf = cross_val_score(lda, X, rng.permutation(y), cv=cv).mean()
    return acc, acc_shuf

rng = np.random.default_rng(1)
WINDOWS = [("target_on", -0.3, 0.0), ("target_on", 0.1, 0.4), ("go_cue", 0.0, 0.3), ("go_cue", 0.3, 0.6),
           ("target_on", -1.3, -1.0)]
rows = []
for s in sessions:
    row = {"session": s["sid"]}
    for a, t0, t1 in WINDOWS:
        acc, sh = decode(s, a, t0, t1, rng)
        row[f"{a}[{t0},{t1}]"] = round(acc, 2)
        row[f"shuffled {a}[{t0},{t1}]"] = round(sh, 2)
    rows.append(row)
dec = pd.DataFrame(rows)
print("Cross-validated decoding of the target (8 targets, chance 0.125) from all units' spike counts:")
print(dec[["session"] + [c for c in dec.columns if c != "session" and not c.startswith("shuffled")]].to_string(index=False))
print("\nShuffled labels, mean over sessions per window:", dec[[c for c in dec.columns if c.startswith("shuffled")]].mean().round(3).to_dict())

In every session the target cannot be decoded before target onset (windows ending at onset) and can
be decoded after it, and best around the go cue, while shuffled labels stay at chance. The spikes
therefore share the clock of the trial table.

## methods-neuron-counts

> Per session, 137 to 256 PMd and 55 to 93 M1 neurons were isolated in Monkey C and 66 to 121 PMd
> and 26 to 51 M1 neurons in Monkey M; neurons above 5 Hz, pooled over all CF and VR sessions,
> number 918 in M1 and 2221 in PMd.

Criterion: units as sorted in the data, per area and session; each count within its monkey's range
widened by 10% of each bound; the 5 Hz rule is the trial-averaged mean rate over all trials of the
session. The files hold only Baseline trials, so the rate is taken over the trials there: the reward
trials from start to stop (main), and every trial of the file (variant). The paper's 16 sessions
cannot be identified, so the two session sets of the data check are both shown.

In [ ]:
rows = []
for s in sessions:
    tr = s["trials"]
    rate_r = perich.mean_rates(s["spikes"], tr[tr.result == "R"])
    rate_all = perich.mean_rates(s["spikes"], tr)
    for area in ("M1", "PMd"):
        m = s["area"] == area
        rows.append({"session": s["sid"], "monkey": s["monkey"], "area": area, "n": int(m.sum()),
                     "above5_reward": int((rate_r[m] > 5).sum()), "above5_all": int((rate_all[m] > 5).sum()),
                     "above1_reward": int((rate_r[m] > 1).sum())})
counts = pd.DataFrame(rows)
wide = counts.pivot(index="session", columns="area", values=["n", "above5_reward", "above5_all"])
print(wide.to_string())

def within(n, lo, hi):
    return (lo - 0.1 * lo) <= n <= (hi + 0.1 * hi)

summary = []
for name, sids in SETS.items():
    c = counts[counts.session.isin(sids)]
    for mk in ("C", "M"):
        for area in ("PMd", "M1"):
            v = c[(c.monkey == mk) & (c.area == area)]
            lo, hi = PAPER[mk][area]
            out = v[~v.n.apply(lambda n: within(n, lo, hi))]
            summary.append({"set": name, "monkey": mk, "area": area, "paper": f"{lo}-{hi}",
                            "data": f"{v.n.min()}-{v.n.max()}", "sessions": len(v),
                            "outside widened range": ", ".join(f"{a} ({b})" for a, b in zip(out.session, out.n)) or "none"})
    for area, exp in (("M1", 918), ("PMd", 2221)):
        c2 = c[c.area == area]
        summary.append({"set": name, "monkey": "pooled", "area": area, "paper": f"{exp} above 5 Hz",
                        "data": f"{c2.above5_reward.sum()} (reward trials), {c2.above5_all.sum()} (all trials)",
                        "sessions": c2.session.nunique(),
                        "outside widened range": ("within 10%" if abs(c2.above5_reward.sum() - exp) <= 0.1 * exp else "outside 10%")
                                                 + " (this set, not 16 sessions; see next cell)"})
print()
print(pd.DataFrame(summary).to_string(index=False))

In [ ]:
# How far the pooled counts depend on which 9 of the Monkey C candidates are the paper's: every choice
# of 9 Monkey C sessions from each set, with the 7 Monkey M sessions.
for name, sids in SETS.items():
    cs = [x for x in sids if x[4] == "C"]; ms = [x for x in sids if x[4] == "M"]
    a5 = counts.set_index(["session", "area"]).above5_reward
    res = []
    for combo in itertools.combinations(cs, 9):
        ss_ = list(combo) + ms
        ok = all(within(int(counts[(counts.session == x) & (counts.area == ar)].n.iloc[0]), *PAPER[x[4]][ar])
                 for x in ss_ for ar in ("M1", "PMd"))
        res.append((sum(a5[(x, "M1")] for x in ss_), sum(a5[(x, "PMd")] for x in ss_), ok))
    res = pd.DataFrame(res, columns=["M1", "PMd", "ranges_hold"])
    both = res[(abs(res.M1 - 918) <= 91.8) & (abs(res.PMd - 2221) <= 222.1)]
    print(f"{name}: {len(res)} choices of 9 C sessions; pooled M1 {res.M1.min()}-{res.M1.max()}, PMd {res.PMd.min()}-{res.PMd.max()}; "
          f"ranges hold in {int(res.ranges_hold.sum())}; both pooled counts within 10% in {len(both)}, "
          f"of which ranges also hold in {int(both.ranges_hold.sum())}")

**Result: partially reproduced, resting on an assumption about which sessions are the paper's.**
Monkey M's seven sessions give 66 to 121 PMd and 26 to 52 M1 units, the paper's ranges (52 is
within 10% of 51). Thirteen of the 14 Monkey C candidates lie inside the widened ranges (PMd 137 to
258, M1 55 to 95); C 2016-09-29 has 114 PMd units, below 137 - 13.7. The ranges hold for 715 of the
2002 ways to choose 9 Monkey C sessions from all 21 candidates, those that leave out C 2016-09-29.
The pooled counts above 5 Hz do not hold under any choice of 9 Monkey C sessions with the 7 Monkey M
sessions: at most 634 M1 and 1224 PMd units (paper 918 and 2221). Even all 21 sessions give 835 M1
(within 10% of 918) and 1565 PMd (outside). Rates over every trial of the file give the same counts
within 10 units. Monkey M has few PMd units above 5 Hz (6 to 21 per session). The pooled verdicts
in the first table are for each whole set, not for 16 sessions; the second cell gives the test on
16 sessions.

## fig3b-pmd-higher-dimensional

> In every session the PMd manifold had more dimensions than the M1 manifold (median 14 against 8.5).

Method from STAR Methods, Dimensionality reduction, paragraph 2 (helper `dimensionality`): Baseline
reward trials, rates as M2, trial-averaged per target; noise from differences between random pairs
of trials to the same target, 1000 repetitions, the 99% limit of each noise eigenvalue. Readings:
threshold rule (a) or (b) (O9), window (a) target onset to acquisition or (b) the whole trial (O3),
kernel width as a standard deviation or a full width at half maximum (O4), and the scale of the
noise, which the paper does not give (raw differences, the text's wording; difference / sqrt 2;
difference / sqrt(2K), the noise of an average of K trials). All units of each area are used.
The claims file expected the kernel reading (O4) not to change an outcome; here it does, and the
results are shown under both. The paper's 16 sessions are 9 of the 14 Monkey C candidates and the 7
Monkey M sessions; the medians are also computed for every such choice.

In [ ]:
t0 = time.time()

def dims_for(i):
    s = sessions[i]
    rng = np.random.default_rng([3, i])   # one generator per session, independent of thread order
    out_fits, out_rows = {}, []
    for window in ("a", "b"):
        for kern in ("sd", "fwhm"):
            fit = perich.latent_fit(s, window, perich.KERNEL_SD_S[kern])
            out_fits[(s["sid"], window, kern)] = fit
            tids = fit["trials"].target_id.to_numpy()
            for area in ("M1", "PMd"):
                d = perich.dimensionality(fit[area]["rates"], tids, rng)
                for scale, (da, db) in d.items():
                    out_rows.append({"session": s["sid"], "monkey": s["monkey"], "window": window, "kernel": kern,
                                     "scale": scale, "area": area, "rule_a": da, "rule_b": db})
    return out_fits, out_rows

fits, rows = {}, []
with ThreadPoolExecutor(4) as ex:
    for f_, r_ in ex.map(dims_for, range(len(sessions))):
        fits.update(f_); rows += r_
dims = pd.DataFrame(rows)
print(f"{len(dims)} dimensionality estimates in {time.time() - t0:.0f} s")
long = dims.melt(id_vars=["session", "monkey", "window", "kernel", "scale", "area"], value_vars=["rule_a", "rule_b"],
                 var_name="rule", value_name="dim")
piv = long.pivot_table(index=["rule", "scale", "window", "kernel", "session"], columns="area", values="dim").reset_index()
rows = []
for name, sids in SETS.items():
    for key, g in piv[piv.session.isin(sids)].groupby(["rule", "scale", "window", "kernel"]):
        rows.append({"set": name, "rule": key[0], "scale": key[1], "window": key[2], "kernel": key[3],
                     "M1_median": g.M1.median(), "PMd_median": g.PMd.median(),
                     "M1_range": f"{g.M1.min()}-{g.M1.max()}", "PMd_range": f"{g.PMd.min()}-{g.PMd.max()}",
                     "PMd>M1": f"{int((g.PMd > g.M1).sum())}/{len(g)}",
                     "medians_in_tol": bool(abs(g.M1.median() - 8.5) <= 0.85 and abs(g.PMd.median() - 14) <= 1.4)})
dimsum = pd.DataFrame(rows)
print(dimsum.to_string(index=False))

In [ ]:
main = piv[(piv.rule == "rule_b") & (piv.scale == "sqrt2K")]
print("Rule (b), noise of the trial average: per session, window (a) and (b), standard-deviation kernel")
show = main[main.kernel == "sd"].pivot(index="session", columns="window", values=["M1", "PMd"])
print(show.to_string())
print("\nRule (b), noise of the trial average: per session, window (a) and (b), full-width kernel")
print(main[main.kernel == "fwhm"].pivot(index="session", columns="window", values=["M1", "PMd"]).to_string())

# The paper used 9 Monkey C sessions and the 7 Monkey M sessions: every choice of 9 of the 14 C candidates
c_sids = [x for x in SETS["all 21"] if x[4] == "C"]; m_sids = [x for x in SETS["all 21"] if x[4] == "M"]
CHOICES = [list(c) + m_sids for c in itertools.combinations(c_sids, 9)]
rows = []
for (w, k), g in main.groupby(["window", "kernel"]):
    g = g.set_index("session")
    med = np.array([[g.loc[ss, "M1"].median(), g.loc[ss, "PMd"].median()] for ss in CHOICES])
    every = np.array([bool((g.loc[ss, "PMd"] > g.loc[ss, "M1"]).all()) for ss in CHOICES])
    tol = (np.abs(med[:, 0] - 8.5) <= 0.85) & (np.abs(med[:, 1] - 14) <= 1.4)
    rows.append({"window": w, "kernel": k, "choices": len(CHOICES), "M1_median_range": f"{med[:, 0].min()}-{med[:, 0].max()}",
                 "PMd_median_range": f"{med[:, 1].min()}-{med[:, 1].max()}", "PMd>M1_in_every_session": int(every.sum()),
                 "medians_in_tol": int(tol.sum()), "both": int((tol & every).sum())})
print("\nRule (b), noise of the trial average, over the", len(CHOICES), "choices of 9 Monkey C sessions with the 7 Monkey M sessions:")
print(pd.DataFrame(rows).to_string(index=False))
for mk in ("C", "M"):
    g = main[(main.kernel == "sd") & (main.session.str[4] == mk)]
    for w in ("a", "b"):
        gw = g[g.window == w]
        print(f"Monkey {mk}, window {w}: M1 median {gw.M1.median()}, PMd median {gw.PMd.median()}, PMd > M1 in {int((gw.PMd > gw.M1).sum())}/{len(gw)}")
fig, axes = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for ax, w in zip(axes, ("a", "b")):
    g = main[(main.kernel == "sd") & (main.window == w)]
    bins = np.arange(0, max(g.PMd.max(), g.M1.max()) + 2) - 0.5
    ax.hist(g.M1, bins, alpha=0.6, label="M1", color="k"); ax.hist(g.PMd, bins, alpha=0.6, label="PMd", color="r")
    ax.set_title(f"window ({w}), rule (b), noise of the average"); ax.set_xlabel("dimensionality"); ax.legend()
axes[0].set_ylabel("sessions")
plt.tight_layout(); plt.show()
print("All values are inside the axes.")

In [ ]:
# Check of `dimensionality` on synthetic data of known rank: 8 targets, 20 bins, 60 neurons, a rank-5
# trial-averaged signal and independent single-trial noise, 25 trials per target.
rng = np.random.default_rng(31)
rows = []
for noise_sd in (0.5, 1.0, 2.0):
    for rep_ in range(3):
        sig = rng.normal(size=(8 * 20, 5)) @ rng.normal(size=(5, 60))
        sig = sig.reshape(8, 20, 60)
        trials, tids = [], []
        for tg in range(8):
            for _ in range(25):
                trials.append(sig[tg] + noise_sd * rng.normal(size=(20, 60))); tids.append(tg)
        d = perich.dimensionality(trials, tids, rng)
        rows.append({"noise_sd": noise_sd, "rep": rep_, **{f"{k}_rule_{r}": v[j] for k, v in d.items() for j, r in enumerate("ab")}})
print("Estimated dimensionality of rank-5 synthetic data, by scale of the noise and rule:")
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced, under one reading and resting on an assumption about the scale of the
noise.** With rule (b), the noise of the trial average, window (a) and a kernel of 100 ms full width
at half maximum, the medians over all 21 sessions are 9 for M1 and 13 for PMd (paper 8.5 and 14,
both within tolerance) and PMd exceeds M1 in 21 of 21 sessions (19 of 19 in the second session
set). Over the 2002 choices of 16 sessions (9 Monkey C with the 7 Monkey M), this reading gives
medians of 8 to 9 (M1) and 12 to 13 (PMd), within tolerance in 910 choices, and PMd above M1 in
every session in all 2002. With a kernel of 100 ms standard deviation the medians are 7 and 10
over 21 sessions (outside tolerance, and outside in every one of the 2002 choices), still with PMd
above M1 in 21 of 21; with window (b), PMd exceeds M1 in 20 of 21 and the medians are 6 to 8. The
outcome therefore rests on the full-width reading of the kernel, on the noise scale, and on the use
of all 21 candidate sessions, as fewer than half of the possible 16-session sets give medians within
tolerance. With one kernel reading for every claim (the standard deviation, adopted below in "One
kernel reading for every claim"), this claim would be partially reproduced: PMd exceeds M1 in every
session and the medians are outside tolerance, over 21 sessions and in all 2002 choices of 16. The
verified outcome is kept, and the change is proposed for the next verification round.
Rule (a) gives medians of 26 to 47 for M1 and 46 to 78 for PMd. With the noise taken as the raw
difference of two trials, or that difference over the square root of two, no component exceeds
the noise and every estimate is 0. On synthetic data of rank 5, rule (b) with the noise of the
trial average returns 5 in all nine runs, while rule (a) with the same scale returns 16 to 45,
which is why rule (a) gives large values here; the raw and root-two scales also return 5 (4 once)
on these synthetic data, whose signal is strong against the single-trial noise, so the synthetic
check does not rule them out in general, only shows that the adopted scale with rule (b) recovers
a known rank.

## figs4d-dimensionality-robust-to-size

> With random subsamples of equal, fixed numbers of neurons, estimated dimensionality fell for small
> samples but PMd stayed higher-dimensional than M1 for every neuron count tested.

As fig3b-pmd-higher-dimensional on random subsamples of 40, 50, 60 and 70 neurons per area and all
neurons, mean over sessions; sessions with fewer neurons than a size are left out of that size.
Run with the noise of the trial average, under both kernel readings (the full-width kernel is the
one whose fig3b medians match the paper) and both windows; both rules are reported. 100 subsamples
per size (the claims file's standard choice), with 100 noise repetitions per subsample to fit the
time limit (the paper's 1000 are kept for "all"); with 100 repetitions the 99th percentile is close
to the largest of them, which can only raise the noise limit slightly.

In [ ]:
t0 = time.time()
SIZES = (40, 50, 60, 70)
N_SUB, SUB_REPS = 100, 100

def subsample_for(i):
    s = sessions[i]
    rng = np.random.default_rng([4, i])
    out = []
    for window, kern in itertools.product(("a", "b"), ("sd", "fwhm")):
        fit = fits[(s["sid"], window, kern)]
        tids = fit["trials"].target_id.to_numpy()
        for area in ("M1", "PMd"):
            n_units = len(fit[area]["units"])
            for size in SIZES:
                if n_units < size:
                    continue
                for k in range(N_SUB):
                    sub = rng.choice(n_units, size, replace=False)
                    d = perich.dimensionality([r[:, sub] for r in fit[area]["rates"]], tids, rng, reps=SUB_REPS)
                    out.append({"session": s["sid"], "window": window, "kernel": kern, "area": area, "size": size,
                                "rule_a": d["sqrt2K"][0], "rule_b": d["sqrt2K"][1]})
    return out

rows = []
with ThreadPoolExecutor(4) as ex:
    for r_ in ex.map(subsample_for, range(len(sessions))):
        rows += r_
sub = pd.DataFrame(rows)
print(f"{len(sub)} subsample estimates in {time.time() - t0:.0f} s")
full = dims[dims.scale == "sqrt2K"].assign(size="all")[["session", "window", "kernel", "area", "size", "rule_a", "rule_b"]]
per_sess = pd.concat([sub.groupby(["session", "window", "kernel", "area", "size"])[["rule_a", "rule_b"]].mean().reset_index(), full])
per_sess["size"] = per_sess["size"].astype(str)
for name, sids in SETS.items():
    g = per_sess[per_sess.session.isin(sids)]
    for rule in ("rule_b", "rule_a"):
        t = g.groupby(["kernel", "window", "size", "area"])[rule].agg(["mean", "sem", "count"]).unstack("area")
        t[("diff", "PMd-M1")] = t[("mean", "PMd")] - t[("mean", "M1")]
        print(f"\nset '{name}', {rule}: mean and s.e.m. over sessions, and the number of sessions")
        print(t.round(2).to_string())

In [ ]:
# Same comparison on the sessions that have both areas at each size (paired), rule (b)
for name, sids in SETS.items():
    g = per_sess[per_sess.session.isin(sids)]
    for k, w in itertools.product(("sd", "fwhm"), ("a", "b")):
        for size in [str(x) for x in SIZES] + ["all"]:
            p = g[(g.kernel == k) & (g.window == w) & (g["size"] == size)].pivot(index="session", columns="area", values="rule_b").dropna()
            if len(p):
                print(f"set '{name}', {k} kernel, window {w}, size {size}: {len(p)} sessions with both areas, "
                      f"PMd > M1 in {int((p.PMd > p.M1).sum())}, mean PMd {p.PMd.mean():.2f}, M1 {p.M1.mean():.2f}")

**Result: reproduced, resting on the same assumption as fig3b-pmd-higher-dimensional.** Under rule (b)
the mean PMd dimensionality exceeds the mean M1 dimensionality at every size (40, 50, 60, 70 and
all neurons) under both kernels, both windows and both session sets, by 1.30 to 4.63. With the
full-width kernel and window (a), the reading whose fig3b medians match the paper, the difference
is 1.88 at 40 neurons and 4.52 with all neurons; with the standard-deviation kernel and window (a),
1.62 and 3.48. Among sessions with both areas at a size, PMd is higher in every session except one
at "all" in window (b) under both kernels (20 of 21) and one at 40 neurons with the full-width
kernel in window (b) (16 of 17). Rule (a) gives PMd above M1 at every size except 40 neurons with
the full-width kernel in window (b) (differences -0.13 and -0.30 in the two sets). The PMd mean
falls at small sizes in every reading (window a, full-width kernel, 13.05 with all neurons against
9.90 at 40; standard-deviation kernel 10.57 against 8.39). The M1 mean does not fall consistently:
in window (a) with the full-width kernel it is 8.52 with all neurons, 8.53 and 8.65 at 60 and 70,
and 8.03 at 40; with the standard-deviation kernel 7.10, 7.12 and 7.20, and 6.77 at 40; in window
(b) with the full-width kernel it is higher at 40 (6.74) than with all neurons (6.52). The
statement's fall for small samples is therefore shown for PMd only; it is outside the criterion.
The means at 40 neurons cover 17 sessions for M1, against 21 for all neurons. The M1 means at
50 to 70 neurons come from fewer sessions (11 to 15), as Monkey M has 26 to 52 M1 units.

## figs4c-null-potent-index-centered

> Single neurons contributed to both output-potent and output-null subspaces; the null/potent index
> was centered on zero with virtually no neurons near -1 or 1.

Index per PMd neuron from the product of the PCA loading matrix (16 PMd components) and V (M3, O7),
as (sum of absolute potent weights - sum of absolute null weights) / (sum of both); the sum of
squares is the second reading. Reproduced when the mean over neurons lies between -0.1 and 0.1 and
fewer than 1% of neurons have an absolute index above 0.9. All PMd units of the candidate sessions
are pooled. The control shuffles the potent weights across neurons against the null weights, 1000
times per session (STAR Methods, "randomly shuffling the weights across neurons").

In [ ]:
rng = np.random.default_rng(5)
rows, pooled = [], {}
for (sid, window, kern), fit in fits.items():
    for power in (1, 2):
        idx = perich.null_potent_index(fit["PMd"]["axes"], fit["potent"], fit["null"], power)
        pooled.setdefault((window, kern, power), {})[sid] = idx
summ = []
for name, sids in SETS.items():
    for (window, kern, power), d in pooled.items():
        v = np.concatenate([d[x] for x in sids])
        summ.append({"set": name, "window": window, "kernel": kern, "magnitude": "abs" if power == 1 else "squares",
                     "neurons": len(v), "mean": round(v.mean(), 3), "sd": round(v.std(), 3),
                     "median": round(np.median(v), 3), "frac_abs>0.9": round((np.abs(v) > 0.9).mean(), 4),
                     "min": round(v.min(), 3), "max": round(v.max(), 3),
                     "meets": bool(-0.1 <= v.mean() <= 0.1 and (np.abs(v) > 0.9).mean() < 0.01)})
print(pd.DataFrame(summ).to_string(index=False))
v = np.concatenate(list(pooled[("a", "fwhm", 2)].values()))
print(f"\nWindow (a), full-width kernel, squares, all 21 sessions: mean {v.mean():.6f}")
for mk in ("C", "M"):
    for key in (("a", "sd", 1), ("a", "fwhm", 1), ("a", "fwhm", 2)):
        v = np.concatenate([x for sid, x in pooled[key].items() if sid[4] == mk])
        print(f"Monkey {mk}, window {key[0]}, {key[1]} kernel, {'abs' if key[2] == 1 else 'squares'}: {len(v)} neurons, "
              f"mean {v.mean():.3f}, fraction |index| > 0.9 {(np.abs(v) > 0.9).mean():.4f}")

# Shuffle control for the main reading (window a, sd kernel, sums of absolute values)
sh_means, sh_frac, sh_sd = [], [], []
weights = [(np.abs(fits[(s["sid"], "a", "sd")]["PMd"]["axes"] @ fits[(s["sid"], "a", "sd")]["potent"]).sum(1),
            np.abs(fits[(s["sid"], "a", "sd")]["PMd"]["axes"] @ fits[(s["sid"], "a", "sd")]["null"]).sum(1)) for s in sessions]
for k in range(1000):
    vals = []
    for wp, wn in weights:
        wp = rng.permutation(wp)
        vals.append((wp - wn) / (wp + wn))
    v = np.concatenate(vals)
    sh_means.append(v.mean()); sh_frac.append((np.abs(v) > 0.9).mean()); sh_sd.append(v.std())
print(f"\nShuffled control (window a, sd kernel, absolute values, 21 sessions, 1000 shuffles): mean {np.mean(sh_means):.3f} "
      f"(2.5-97.5%: {np.percentile(sh_means, 2.5):.3f} to {np.percentile(sh_means, 97.5):.3f}), sd {np.mean(sh_sd):.3f}, "
      f"fraction |index| > 0.9: {np.mean(sh_frac):.4f}")
per = pd.DataFrame([{"session": sid, "mean": round(v.mean(), 3), "sd": round(v.std(), 3), "n": len(v)}
                    for sid, v in pooled[("a", "sd", 1)].items()])
print("\nPer session, main reading:"); print(per.to_string(index=False))
v = np.concatenate(list(pooled[("a", "sd", 1)].values()))
plt.figure(figsize=(4, 3)); plt.hist(v, np.linspace(-1, 1, 41), color="k")
plt.xlabel("null/potent index"); plt.ylabel("PMd neurons"); plt.title("window (a), sd kernel, all 21 sessions")
plt.tight_layout(); plt.show()
print(f"Index range {v.min():.3f} to {v.max():.3f}; all points inside the axes.")

**Result: reproduced, resting on the reading of "magnitude" as the sum of absolute values.** Over
3263 PMd units of the 21 sessions, the mean index is 0.017 to 0.056 across the four window and kernel
readings, and no unit has an absolute index above 0.9 (range -0.701 to 0.741). With sums of squares,
three of four readings also meet the criterion; window (a) with the full-width kernel gives a mean
of 0.100085, just above the bound of 0.1 (0.09% of units above 0.9). The shuffled control has a
similar mean (0.036) and a wider spread (standard deviation 0.483 against 0.217, 3.2% of units
above 0.9). The index leans positive in Monkey M: its session means are 0.081 to 0.143 under the
main reading, and its 599 pooled units have a mean of 0.109 (above the bound of 0.1 if Monkey M
were judged alone), against 0.028 for the 2664 Monkey C units (session means -0.007 to 0.081).

## fig6c-null-leads-potent

> In Baseline, output-null activity preceded output-potent activity by 35 ms on average across the 16
> sessions, a lag significantly below zero (t-test, p = 0.02), and output-null also led at the end
> of Force.

Method from STAR Methods, Calculating the lag (helper `cca_lag`): Baseline trials, the 700 ms after
target presentation, potent activity shifted from -350 to 350 ms in 10 ms steps against null
activity, CCA at each lag, the lag of the maximum first canonical correlation per session; trials
concatenated within a session. Readings of the time resolution: (a) 10 ms bins smoothed as M2,
(b) 50 ms latent activity interpolated to 10 ms. The subspaces come from the PCA and W of the
window readings (a) and (b) (O3), under both kernel readings (O4). One-sample two-sided t-test of
the session lags against zero. A maximum within one step of the edge of the search (|lag| of 340 or
350 ms) is flagged, and the test is shown with such sessions kept and dropped. As the paper's test
is over 16 sessions, it is also run on every choice of 9 of the 14 Monkey C candidates with the 7
Monkey M sessions. The end-of-Force part cannot be tested: no Force trials are in the dandiset.

In [ ]:
t0 = time.time()

def lags_for(s):
    out_rows, out_curves = [], {}
    for window in ("a", "b"):
        for kern in ("sd", "fwhm"):
            fit = fits[(s["sid"], window, kern)]
            onsets = fit["trials"].target_on.to_numpy()
            for res in ("a", "b"):
                lat = perich.lag_grid_latents(s, fit["PMd"]["units"], onsets, fit["PMd"]["axes"], res,
                                              perich.KERNEL_SD_S[kern])
                lag, lags, ccs = perich.cca_lag([l @ fit["null"] for l in lat], [l @ fit["potent"] for l in lat])
                out_curves[(s["sid"], window, kern, res)] = ccs
                out_rows.append({"session": s["sid"], "monkey": s["monkey"], "window": window, "kernel": kern,
                                 "resolution": res, "lag_ms": round(lag * 1000), "peak_cc": round(ccs.max(), 3),
                                 "cc_at_0": round(ccs[np.argmin(np.abs(lags))], 3)})
    return out_rows, out_curves

rows, curves = [], {}
with ThreadPoolExecutor(4) as ex:
    for r_, c_ in ex.map(lags_for, sessions):
        rows += r_; curves.update(c_)
lagdf = pd.DataFrame(rows)
# A maximum at the last or second-to-last lag of the search (|lag| >= 340 ms) is an edge maximum: the
# correlation may still rise outside the range, so the lag is not located. Shown with and without them.
lagdf["edge"] = lagdf.lag_ms.abs() >= 340
print(f"{len(lagdf)} lag estimates in {time.time() - t0:.0f} s")
print("Edge maxima:", lagdf[lagdf.edge][["session", "window", "kernel", "resolution", "lag_ms"]].to_dict("records"))
summ = []
for (name, sids), drop_edge in itertools.product(SETS.items(), (False, True)):
    d_ = lagdf[lagdf.session.isin(sids) & ~(lagdf.edge & drop_edge)]
    for key, g in d_.groupby(["window", "kernel", "resolution"]):
        t = stats.ttest_1samp(g.lag_ms, 0)
        summ.append({"set": name, "edge_maxima": "dropped" if drop_edge else "kept", "window": key[0], "kernel": key[1], "resolution": key[2], "sessions": len(g),
                     "mean_lag_ms": round(g.lag_ms.mean(), 1), "sem": round(g.lag_ms.sem(), 1),
                     "median": g.lag_ms.median(), "negative": int((g.lag_ms < 0).sum()), "zero": int((g.lag_ms == 0).sum()),
                     "t": round(t.statistic, 2), "p": round(t.pvalue, 4),
                     "within_-35+-3.5": bool(abs(g.lag_ms.mean() + 35) <= 3.5)})
print(pd.DataFrame(summ).to_string(index=False))

In [ ]:
print("Per session lags (ms), standard-deviation kernel:")
print(lagdf[lagdf.kernel == "sd"].pivot(index="session", columns=["window", "resolution"], values="lag_ms").to_string())
for mk in ("C", "M"):
    g = lagdf[(lagdf.monkey == mk) & (lagdf.kernel == "sd")]
    print(f"Monkey {mk}:", g.groupby(["window", "resolution"]).lag_ms.agg(["mean", "count"]).round(1).to_dict("index"))
fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True)
for ax, (w, res) in zip(axes.ravel(), itertools.product(("a", "b"), ("a", "b"))):
    for s in sessions:
        ax.plot(perich.LAGS_S * 1000, curves[(s["sid"], w, "sd", res)], lw=0.8)
    ax.axvline(0, color="k", lw=0.5); ax.set_title(f"window ({w}), resolution ({res}), sd kernel, 21 sessions")
    allc = np.concatenate([curves[(s["sid"], w, "sd", res)] for s in sessions])
    print(f"window ({w}), resolution ({res}): plotted correlations range {allc.min():.3f} to {allc.max():.3f}; all inside the axes.")
for ax in axes[1]:
    ax.set_xlabel("lag of potent relative to null (ms; negative: null leads)")
for ax in axes[:, 0]:
    ax.set_ylabel("first canonical correlation")
plt.tight_layout(); plt.show()

In [ ]:
# The paper's t-test is over 16 sessions (9 Monkey C, 7 Monkey M): the same test on every choice of
# 9 of the 14 Monkey C candidates with the 7 Monkey M sessions, per reading, edge maxima kept or dropped.
rows = []
sig_any = {False: np.zeros(len(CHOICES), bool), True: np.zeros(len(CHOICES), bool)}
for (w, k, res), g in lagdf.groupby(["window", "kernel", "resolution"]):
    g = g.set_index("session")
    for drop_edge in (False, True):
        ps, means = [], []
        for ss in CHOICES:
            v = g.loc[ss]
            v = v[~v.edge] if drop_edge else v
            ps.append(stats.ttest_1samp(v.lag_ms, 0).pvalue); means.append(v.lag_ms.mean())
        ps, means = np.array(ps), np.array(means)
        sig = (ps < 0.05) & (means < 0)
        sig_any[drop_edge] |= sig
        rows.append({"window": w, "kernel": k, "resolution": res, "edge_maxima": "dropped" if drop_edge else "kept",
                     "mean_lag_range_ms": f"{means.min():.1f} to {means.max():.1f}", "median_p": round(np.median(ps), 3),
                     "significant_negative": f"{int(sig.sum())}/{len(CHOICES)}", "fraction": round(sig.mean(), 2),
                     "within_-35+-3.5_and_significant": int((sig & (np.abs(means + 35) <= 3.5)).sum())})
print(pd.DataFrame(rows).to_string(index=False))
for drop_edge, v in sig_any.items():
    print(f"edge maxima {'dropped' if drop_edge else 'kept'}: significantly negative under at least one of the 8 readings "
          f"in {int(v.sum())} of {len(CHOICES)} choices ({v.mean():.2f})")

**Result: partially reproduced, under one reading, for Baseline only, and resting on the use of all
21 candidate sessions.** The mean Baseline lag over the 21 sessions is negative (output-null first)
under all 8 readings (-2.4 to -51.4 ms). It is significantly below zero only with window (a) and the
100 ms standard-deviation kernel: -18.1 ms (t = -2.29, p = 0.033) at 10 ms resolution and -51.4 ms
(p = 0.035) with interpolated 50 ms activity. In the latter, two sessions peak within one step of the
edge of the search (C 2016-10-06 at -350 ms and C 2016-10-11 at -340 ms); without them the mean is
-20.5 ms over 19 sessions (p = 0.034). The full-width kernel and window (b) give p from 0.18 to 0.82.
No reading over 21 sessions is within 3.5 ms of -35 ms. On the 2002 choices of 16 sessions the
result is fragile: the main reading (window a, standard-deviation kernel, 10 ms) is significantly
negative in 763 (38%), the 50 ms reading in 478 (24%; 816, 41%, with edge maxima dropped), and at
least one of the 8 readings in 1013 (51%; 1052 with edge maxima dropped). Twelve choices, all with
edge maxima dropped at 50 ms resolution, give a significant mean within tolerance of -35 ms. In the
second session set the main reading is significant (p = 0.040). By monkey, window (a) and 10 ms
resolution give -22.9 ms for C and -8.6 ms for M. The end-of-Force part cannot be tested. The
outcome rests on the standard-deviation reading of the kernel, which is the reading adopted for
every claim (below); with the full-width reading no reading is significant (p 0.18 to 0.38), which
would make it not reproduced.

## One kernel reading for every claim

The claims file names two readings of the kernel's "width 100 ms" (O4) and expected them not to
change an outcome. They do, in opposite directions for fig3b-pmd-higher-dimensional and
fig6c-null-leads-potent, so a reading chosen per claim would take the favourable one for each. The
cell below gives, for each kernel reading applied to all claims, the numbers each criterion is
judged on (rule b and the noise of the trial average for the dimensionality; all 21 sessions).

In [ ]:
# Per kernel reading: the fig3b main reading, the figs4c readings, the figs4d comparison and the fig6c t-tests
rows = []
for kern in ("sd", "fwhm"):
    d3 = dimsum[(dimsum.set == "all 21") & (dimsum.rule == "rule_b") & (dimsum.scale == "sqrt2K") & (dimsum.kernel == kern)]
    f3 = "; ".join(f"window {r.window}: medians {r.M1_median}/{r.PMd_median}, PMd>M1 {r['PMd>M1']}, in tolerance {r.medians_in_tol}"
                   for _, r in d3.iterrows())
    meets4c = []
    for (w, k, power), d in pooled.items():
        if k == kern:
            v = np.concatenate(list(d.values()))
            meets4c.append(bool(-0.1 <= v.mean() <= 0.1 and (np.abs(v) > 0.9).mean() < 0.01))
    g4 = per_sess[per_sess.kernel == kern].groupby(["window", "size", "area"]).rule_b.mean().unstack("area")
    lag_sig = []
    for (w, res), g in lagdf[lagdf.kernel == kern].groupby(["window", "resolution"]):
        t = stats.ttest_1samp(g.lag_ms, 0)
        lag_sig.append(f"window {w} res {res}: {g.lag_ms.mean():.1f} ms, p {t.pvalue:.3f}")
    rows.append({"kernel": kern, "fig3b": f3, "figs4c readings meeting": f"{sum(meets4c)}/{len(meets4c)}",
                 "figs4d PMd>M1 at every size and window": bool((g4.PMd > g4.M1).all()), "fig6c": "; ".join(lag_sig)})
for r in rows:
    print(f"Kernel reading '{r['kernel']}':")
    for k, v in r.items():
        if k != "kernel":
            print(f"  {k}: {v}")


**Kernel reading adopted for every claim: a standard deviation of 100 ms.** STAR Methods,
Dimensionality reduction, says only "a non-causal Gaussian kernel of width 100 ms". The protocol
lets the author adopt the reading the paper best supports, as an assumption, when readings disagree
(the claims file's header instead says such a claim is blocked; the protocol's later rule is
followed here, and the conflict is recorded). The standard deviation is better supported: it is the
usual way a Gaussian smoothing kernel's width is stated, and the same group's later paper on
recordings that include Monkey C (Gallego, Perich et al. 2020, Nat. Neurosci., in `paper/`) gives
its kernel as "a Gaussian kernel (s.d.: 50 ms)" for the same square-root, binned rates. Applied to all claims, the standard-deviation reading leaves figs4d, figs4c and
fig6c as recorded, and gives fig3b medians of 7 and 10 with PMd above M1 in every session, which
the criterion counts as partially reproduced. The verified outcome of fig3b (reproduced, on the
full-width reading) is kept, and the change to partially reproduced is proposed for the next
verification round. Under the full-width reading fig3b would be reproduced and no fig6c reading
would be significant (not reproduced).

## Claims not tested

The dandiset holds no Force, Rotation or Washout trials, and no label of which sessions were curl
field or visuomotor rotation sessions (data check). Every claim that needs those trials is blocked
with `data_missing`: fig2d-cf-behavioral-adaptation, fig7bc-vr-behavior-similar,
figs2-covariance-preserved-example, figs3a-covariance-like-baseline-split,
figs3b-covariance-preserved-vr, figs5e-glm-cell-counts, figs5e-pot-null-similar-training,
fig3e-local-rpr2-unchanged, fig3g-local-models-generalize, fig4d-null-m1-fails-early,
fig4c-null-tracks-behavior, figs6-pc-m1-fails, figs6de-per-monkey, figs5f-rpr2-distributions,
figs7a-unnormalized-change, figs7e-dimensionality-range, figs7f-factor-analysis,
figs7cd-single-neuron-models, fig5-washout-null-m1, fig6e-planning-change-in-null,
fig7de-vr-all-generalize, fig7f-vr-washout-unchanged and cf-vr-dissociation.

## Summary

| Claim | Outcome |
|---|---|
| methods-neuron-counts | partially reproduced (assumed: which sessions are the paper's) |
| fig3b-pmd-higher-dimensional | reproduced (assumed: scale of the noise; holds with the full-width kernel only, and in 910 of 2002 sets of 16 sessions); partially reproduced proposed, under the kernel reading adopted for every claim |
| figs4d-dimensionality-robust-to-size | reproduced (assumed: scale of the noise) |
| figs4c-null-potent-index-centered | reproduced (assumed: magnitude as absolute values) |
| fig6c-null-leads-potent | partially reproduced, Baseline only (standard-deviation kernel, all 21 candidate sessions; significant in 38% of 16-session choices) |
| 23 claims on Force, Rotation and Washout trials | blocked, data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

**How the analysis differs from the paper's.** The paper's 16 sessions cannot be identified; all 21
dual-array center-out sessions of Monkeys C and M are used, with the 19 whose reward counts fit the
paper's Baseline range as a second set, and every choice of 16 sessions (9 of the 14 Monkey C
candidates with the 7 Monkey M sessions) for the session-level tests. All of each file's reward
trials are taken as Baseline. Target acquisition is the trial's stop time minus the 0.5 s hold. The
scale of the noise in the dimensionality estimate is not given; the noise of the trial average is
assumed, as the text's raw differences give no dimensions at all. The kernel width (standard
deviation or full width at half maximum) changes the fig3b medians and the fig6c significance,
which the claims file did not expect, and no single reading gives the recorded outcome of both:
the full-width reading supports fig3b and the standard-deviation reading fig6c. The
standard-deviation reading is adopted for every claim, as the better supported; under it fig3b
would be partially reproduced, a change proposed for the next verification round. Figure S4D uses 100 subsamples per size with 100 noise
repetitions per subsample.

**What was found about the dataset.** It holds only the Baseline epoch of the sessions: about 200
reward trials each, with no perturbation block. The Force or Rotation and Washout epochs, and any
label of a session as curl field or visuomotor rotation, are missing, so 23 of the 28 claims cannot
be tested. Nothing marks which sessions are the paper's. In seven Monkey C sessions the session
description ends in a cut-off warning, and the cursor ends reaches at about 4.2 cm while the
targets are stored at 8 cm; in two Monkey C sessions the cursor timestamps repeat thousands of
times; in several sessions the cursor never enters the stored target square. The pooled counts of
units above 5 Hz are far below the paper's.

In [ ]:
print(f"notebook ran in {(time.time() - started) / 60:.1f} min")